# Deliverable 3 - INF201 

Submission deadline: 13th Nov 2026

### Hand-in information:
* This is a group submission, so one person submits on behalf of the pair/group
* Submit Python scripts (.py), a Jupyter notebook (ipynb) <i>and</i> a pdf of the same notebook
    * See Overview and objectives for a list of what your notebook should contain
* Include a link to your GitHub repository and a log of the commits
    * To generate the log, paste this into the terminal after your last commit:<br>
    `git log --graph --pretty=format:"%h - %an, %ar : %s" > git_log.txt`
    * The repository should contain a `requirements.txt` with information on your virtual environment
    * Note: Remember to create a new virtual environment for each deliverable
* Include a discussion of GenAI use, for example a summary of the chat log as you have
done earlier
    * This can be a separate file or be included in the Jupyter notebook
* Make a short (5 mins) video discussing your code and results
    * Note that you both have to contribute to the video
    * Either by recording it together
    * Or recording parts separately and editing it together
* We would appriciate it if use English for this assignment, both in the writing and in the video

## Overview and objectives

In this assignment you will continue working on the Character class in `game_simulator_v1.py` that we created in lecture_08 (check out the Week 44 module), and expand this to create a mini-RPG in the form of a battle simulator. To do this you will update and extend the existing class and methods, introduce new classes allowing us to use dice and weapons, and create a combat manager class.

The rules for the game is as follows:
* The game you build should allow two characters to battle until one of them wins. 
* The characters can use weapons to attack, where the damage dealt is determined by a dice roll.
* The characters can have different specialisations (warrior or mage). 
* A character's HP (hit points, or health) must be between 0 and the max given for this specific character.
* The character dies when it reaches 0 HP.

Learning Objectives:
* Enforce data encapsulation using private attributes and properties.
* Implement custom classes to model discrete concepts (`Dice`, `Weapon`, `Battle`).¨
* Create subclasses inheriting from the `Character` class.
* Apply object composition by equipping characters with weapons and dice.

To organise your project, we recommend the following struckture:
* Separate modules (Python scripts) for each of the classes created
* A Jupyter Notebook containing the following:
    * A presentation of each class created (showing attributes and methods objects of the class has)
    * Simulated battles between different types of characters, with and without weapons

   ## Setup
   The classes are written in separate Python modules (`.py` files) and imported here. All imports are collected in this cell, so the notebook can be run from top to bottom.

In [1]:
%load_ext autoreload
%autoreload 2

import random
from collections import Counter

from character import Character, Mage, Warrior
from dice import Dice

from items import Weapon
from battle import Battle

## Task 1: Update the Character class

When we created the Character class in the lecture we first included an `attack()` method, and afterwards indicated that we wanted hit points to be a protected feature `_hp`. This lead to the `Character.attack()` method changing the target's protected attribute. Modifying another object's private attribute is bad form! Your first task is to fix this.

* Implement a `take_damage()` method to be used when the `Character` object is injured. This method should only affect the characters *own* HP.
* Implement a `heal()` method to be used when the `Character` heals (in other words increase their hit points)
* Change the existing `attack()` method so that calls the target's `take_damage()` method rather than directly modifying the target's `_hp`.
* Add an `is_alive` property to the `Character` class. This should be read-only, and return True if `_hp`$ > 0$ and False otherwise.

### Task 1 – Changes to the Character class
* `take_damage(amount)` reduces the character's **own** `_hp`. `max(..., 0)` makes sure HP never goes below 0. A negative amount is set to 0, so taking damage can never heal the character.
* `heal(amount)` increases the character's own `_hp`. `min(..., hp_max)` makes sure HP never goes above the maximum. A negative amount is set to 0, so healing can never hurt.
* `attack(target)` now calls `target.take_damage(self.damage)` instead of changing `target._hp` directly. Each character is responsible for its own HP, which is the idea behind encapsulation.
* `is_alive` is a read-only property that returns `True` if `_hp > 0`. It has no setter, so it cannot be changed from outside the class.

Together, `take_damage()` and `heal()` make sure HP always stays between 0 and `hp_max`, as the rules of the game require.

In [2]:
hero = Character("Hero")
enemy = Character("Goblin", hp_max=25, damage=5)

print(hero)
print(enemy)
print(repr(hero))

# Attributes and methods of a Character object (special methods like __init__ are left out)
print("\nAttributes and methods:", [name for name in dir(hero) if not name.startswith("__")])

# The hero attacks until the goblin is dead
print()
while enemy.is_alive:
    hero.attack(enemy)
    print(f"{hero.name} attacks {enemy.name}: {enemy.name} has {enemy.hp} HP, alive: {enemy.is_alive}")

# Healing cannot go above hp_max
hero.take_damage(30)
print(f"\n{hero.name} takes 30 damage: {hero.hp} HP")
hero.heal(50)
print(f"{hero.name} heals 50: {hero.hp} HP (max is {hero.hp_max})")

Character: Hero with 100 HP and 10 damage
Character: Goblin with 25 HP and 5 damage
<Character name=Hero hp=100 damage=10 weapon=None>

Attributes and methods: ['_hp', 'attack', 'damage', 'heal', 'hp', 'hp_max', 'introduce', 'is_alive', 'name', 'take_damage', 'weapon']

Hero attacks Goblin: Goblin has 15 HP, alive: True
Hero attacks Goblin: Goblin has 5 HP, alive: True
Hero attacks Goblin: Goblin has 0 HP, alive: False

Hero takes 30 damage: 70 HP
Hero heals 50: 100 HP (max is 100)


### Task 1 – Tests
The tests check that HP always stays between 0 and `hp_max`, that `attack()` only changes the target's HP through `take_damage()`, and that `hp` and `is_alive` are read-only.

In [3]:
import inspect

# take_damage: HP goes down, but never below 0, and negative damage does not heal
c = Character("Test", hp_max=50)
c.take_damage(20)
assert c.hp == 30
c.take_damage(100)
assert c.hp == 0 and not c.is_alive
c.take_damage(-5)
assert c.hp == 0

# heal: HP goes up, but never above hp_max, and negative healing does not hurt
c = Character("Test", hp_max=50)
c.take_damage(30)
c.heal(10)
assert c.hp == 30
c.heal(100)
assert c.hp == 50
c.heal(-10)
assert c.hp == 50

# attack: only the target loses HP, and attack() does not touch the target's _hp directly
attacker = Character("Attacker", damage=15)
target = Character("Target")
attacker.attack(target)
assert target.hp == 85 and attacker.hp == 100
assert "_hp" not in inspect.getsource(Character.attack)

# is_alive: True while HP > 0, False at 0
c = Character("Test", hp_max=10)
assert c.is_alive
c.take_damage(10)
assert not c.is_alive

# hp and is_alive are read-only
for attribute in ["hp", "is_alive"]:
    try:
        setattr(c, attribute, 100)
        raise AssertionError(f"{attribute} should be read-only")
    except AttributeError:
        pass

print("All Task 1 tests passed.")

All Task 1 tests passed.


## Task 2: Introducing randomness with a Dice class

Currently, characters deal fixed damage. To introduce variable damage, create a separate `Dice` class.

* Create a `Dice` class that can create any n-sided Dice (n>=2), where the default number of sides is 6.
* Add a `roll()` method to roll the dice, using `random.randint(1, self.sides)`.
* Add a `__repr__` method that returns string representations like `Dice(sides=6)`.
* Implement a `__call__` method that allow you to roll the dice created by `d20 = Dice(sides=20)` by calling `d20.roll()`.
    * See the [Python documentation of `__call__`](https://docs.python.org/3/reference/datamodel.html#object.__call__) or [this example](https://www.w3schools.com/python/python_magic_call.asp)

### Task 2 – The Dice class
The `Dice` class is written in its own module, `dice.py`.
* `Dice(sides=6)` creates a dice with any number of sides. The default is 6. The number of sides is checked when the dice is created: fewer than 2 sides raises a `ValueError`, and a non-integer (for example `6.5` or `"6"`) raises a `TypeError`. This way an invalid dice can never exist.
* The number of sides is stored in the protected attribute `_sides`, and shown through the read-only property `sides`. This follows the same idea as `hp` in the `Character` class: the value can be read, but not changed to something invalid after the dice is created.
* `roll()` returns `random.randint(1, self.sides)`, which is a random integer from 1 to `sides`, where both ends are included.
* `__repr__` returns a string like `Dice(sides=6)`. This is also valid Python code that recreates the same dice.
* `__call__` makes the object callable, so `d20()` rolls the dice. It simply returns `self.roll()`, so `d20()` and `d20.roll()` always do the same thing. (The task text writes `d20.roll()`, but it is `__call__` that makes the shorter `d20()` possible.)

In the demo below we set a fixed random seed with `random.seed()`, so the notebook gives the same output every time it is run.

In [4]:
random.seed(201)  # Fixed seed, so the output is the same every time the notebook is run

d6 = Dice()           # Default: 6 sides
d20 = Dice(sides=20)

print(d6)             # print() falls back to __repr__ because Dice has no __str__
print(repr(d20))
print(f"d6 has {d6.sides} sides and d20 has {d20.sides} sides")

# Attributes and methods of a Dice object (special methods like __init__ are left out)
print("\nAttributes and methods:", [name for name in dir(d20) if not name.startswith("__")])

# roll() and __call__ do the same thing
print("\nFive rolls with d20.roll():", [d20.roll() for _ in range(5)])
print("Five rolls with d20():     ", [d20() for _ in range(5)])

# Roll the six-sided dice many times to see that all sides come up about equally often
rolls = Counter(d6() for _ in range(6000))
print("\n6000 rolls with d6 (expected about 1000 of each):")
for side in range(1, d6.sides + 1):
    print(f"  {side}: {rolls[side]}")

# Trying to create a dice with only one side
try:
    Dice(sides=1)
except ValueError as error:
    print(f"\nDice(sides=1) -> ValueError: {error}")

Dice(sides=6)
Dice(sides=20)
d6 has 6 sides and d20 has 20 sides

Attributes and methods: ['_sides', 'roll', 'sides']

Five rolls with d20.roll(): [3, 11, 14, 1, 10]
Five rolls with d20():      [11, 16, 20, 9, 15]

6000 rolls with d6 (expected about 1000 of each):
  1: 1032
  2: 983
  3: 953
  4: 1051
  5: 1004
  6: 977

Dice(sides=1) -> ValueError: A dice must have at least 2 sides, got 1


### Task 2 – Tests
The tests check the default number of sides, the `__repr__` string, that every roll is between 1 and `sides`, that all sides can come up, that `d20()` gives the same results as `d20.roll()` with the same seed, that invalid dice cannot be created, and that `sides` is read-only.

In [5]:
# Default dice has 6 sides, and any number of sides >= 2 is allowed
assert Dice().sides == 6
assert Dice(sides=2).sides == 2
assert Dice(sides=100).sides == 100

# __repr__ gives strings like Dice(sides=6)
assert repr(Dice()) == "Dice(sides=6)"
assert repr(Dice(sides=20)) == "Dice(sides=20)"

# roll() and __call__ always return an integer from 1 to sides
for sides in [2, 6, 20]:
    dice = Dice(sides)
    for _ in range(1000):
        roll = dice.roll()
        assert isinstance(roll, int) and 1 <= roll <= sides
        assert 1 <= dice() <= sides

# All sides come up when rolling many times
d6 = Dice()
assert {d6() for _ in range(1000)} == {1, 2, 3, 4, 5, 6}

# d20() and d20.roll() give exactly the same results with the same seed
d20 = Dice(sides=20)
random.seed(42)
rolls_with_roll = [d20.roll() for _ in range(20)]
random.seed(42)
rolls_with_call = [d20() for _ in range(20)]
assert rolls_with_roll == rolls_with_call

# A dice with fewer than 2 sides cannot be created
for invalid_sides in [1, 0, -6]:
    try:
        Dice(invalid_sides)
        raise AssertionError(f"Dice({invalid_sides}) should raise ValueError")
    except ValueError:
        pass

# The number of sides must be an integer
for invalid_sides in [6.5, "6"]:
    try:
        Dice(invalid_sides)
        raise AssertionError(f"Dice({invalid_sides!r}) should raise TypeError")
    except TypeError:
        pass

# sides is read-only
try:
    d6.sides = 1
    raise AssertionError("sides should be read-only")
except AttributeError:
    pass

print("All Task 2 tests passed.")

All Task 2 tests passed.


## Task 3: Create items

Create a `Weapon` class in `items.py` that utilizes your `Dice` class for damage calculations. We call the module `items.py` rather than `weapons.py` so that it can be expanded to include other items your characters can hold.

* Define the class `Weapon` with the attributes `name` and `damage_dice: Dice`.
* Include a `roll_damage()` method that calls `self.damage_dice.roll()`
* Update `Character.__init__` to accept an optional weapon.
* If a character has an equipped weapon, `attack()` should roll damage from the weapon instead of using default base damage.

### Task 3 – The Weapon class
The `Weapon` class is writen in its own module, `items.py`.

* A "Weapon" has a "name" and a "damage_dice", which is a `Dice` object from Task 2. This is an example of object composition: the weapon *has* a dice, and lets the dice do the rolling.
* `roll_damage()` returns `self.damage_dice.roll()`.
* `Character.__init__` now takes an optional `weapon`. The default is `None`, which means the character has no weapon.
* `Character.attack()` checks if the character has a weapon. If it does, the damage is rolled with the weapon's dice. If not, the fixed base `damage` is used. In both cases the damage is passed on to `target.take_damage()`, so the change from Task 1 still holds.

In [6]:
random.seed(201)  # Fixed seed, so the output is the same every time the notebook is run

sword = Weapon("Sword", Dice(sides=12))
print(sword)
print("Attributes and methods:", [name for name in dir(sword) if not name.startswith("__")])
print("Ten rolls with the sword:", [sword.roll_damage() for _ in range(10)])

# A character with a weapon and one without
knight = Character("Knight", weapon=sword)
farmer = Character("Farmer")
print()
print(repr(knight))
print(repr(farmer))

# Attack a dummy with lots of HP 1000 times, and look at the damage dealt
def damage_per_attack(attacker: Character, n_attacks: int = 1000) -> list[int]:
    damages = []
    for _ in range(n_attacks):
        dummy = Character("Dummy")
        attacker.attack(dummy)
        damages.append(dummy.hp_max - dummy.hp)
    return damages

for attacker in [knight, farmer]:
    damages = damage_per_attack(attacker)
    print(f"\n{attacker.name}: min {min(damages)}, max {max(damages)}, "
          f"average {sum(damages) / len(damages):.2f}")

Weapon(name=Sword, damage_dice=Dice(sides=12))
Attributes and methods: ['damage_dice', 'name', 'roll_damage']
Ten rolls with the sword: [2, 12, 6, 7, 1, 5, 6, 12, 12, 11]

<Character name=Knight hp=100 damage=10 weapon=Weapon(name=Sword, damage_dice=Dice(sides=12))>
<Character name=Farmer hp=100 damage=10 weapon=None>

Knight: min 1, max 12, average 6.54

Farmer: min 10, max 10, average 10.00


**Observation.** The sword uses a 12-sided dice, so the average damage is $(1 + 12) / 2 = 6.5$, which matches the result above. This is lower than the base damage of 10. In other words, the knight with a sword deals *less* damage on average than the farmer without a weapon, but the damage varies a lot from attack to attack. To make weapons stronger than fighting without one, they would need a dice with more sides, for example a 20-sided dice (average 10.5).

### Task 3 – Tests
The tests check that a weapon stores its name and dice, that `roll_damage()` uses the dice, that characters have no weapon by default, and that `attack()` uses the weapon's dice when a weapon is equipped and the base damage otherwise.

In [7]:
# A weapon stores its name and its dice
d8 = Dice(sides=8)
axe = Weapon("Axe", d8)
assert axe.name == "Axe" and axe.damage_dice is d8

# roll_damage() gives a value from 1 to the number of sides, and all values can come up
assert all(1 <= axe.roll_damage() <= 8 for _ in range(1000))
assert {axe.roll_damage() for _ in range(1000)} == set(range(1, 9))

# roll_damage() uses the dice: with the same seed it gives the same rolls as the dice itself
random.seed(7)
dice_rolls = [d8.roll() for _ in range(20)]
random.seed(7)
weapon_rolls = [axe.roll_damage() for _ in range(20)]
assert dice_rolls == weapon_rolls

# A character has no weapon by default, and then always deals its base damage
farmer = Character("Farmer", damage=7)
assert farmer.weapon is None
target = Character("Target")
farmer.attack(target)
assert target.hp == 93

# With a weapon, the damage comes from the weapon's dice and not the base damage
knight = Character("Knight", damage=50, weapon=Weapon("Stick", Dice(sides=2)))
for _ in range(100):
    target = Character("Target")
    knight.attack(target)
    assert 1 <= 100 - target.hp <= 2

# The damage dealt is exactly the weapon's roll
sword = Weapon("Sword", Dice(sides=12))
knight = Character("Knight", weapon=sword)
random.seed(5)
expected = [sword.roll_damage() for _ in range(10)]
random.seed(5)
dealt = []
for _ in range(10):
    target = Character("Target")
    knight.attack(target)
    dealt.append(100 - target.hp)
assert dealt == expected

print("All Task 3 tests passed.")

All Task 3 tests passed.


## Task 4: Character specialization

Extend the character module to include two specialised subclasses of `Character`.

1. `Warrior` subclass (defensive specialist)
    * New attribute: `armor: int`, with a default value of 5
    * Override the `Character.take_damage()` method by reducing the incoming damage by armor before applying it.
2. `Mage` subclass (resource-based attacker)
    * New attributes: `mana: int` and `mana_max: int`, both with default value 30
    * Magic attacks consume mana, so the mana available determines whether the mage can use magic.
    * Override the `Character.attack()` method:
        * If the mage has at least 10 mana left, spend 10 mana to deal a magical spell attack dealing double damage. This magic attack does not use any weapons, even if the character has one equipped.
        * If the mage has less than 10 mana left, fall back to the base attack, and restore 5 mana. This base attack should use whichever weapon the character has equipped.

### Task 4 – The Warrior and Mage subclasses
Both classes are written in `character.py` and inherit from `Character`. This means they get all of `Character`'s attributes and methods (`hp`, `heal()`, `is_alive`, the weapon, and so on), and only add or override what is special for them. Both use `super().__init__()`, so the setup of name, HP, damage and weapon is not repeated.

**Warrior**
* New attribute `armor`, with default value 5. Negative armor raises a `ValueError`.
* `take_damage()` is overridden. It subtracts `armor` from the incoming damage and passes the rest on to `Character.take_damage()` with `super()`. If the damage is smaller than the armor, the result is negative, and `Character.take_damage()` already turns negative damage into 0. A weak hit therefore does no damage, but it can never heal the warrior.
* Because `Character.attack()` calls `target.take_damage()`, the armor works automatically no matter who attacks the warrior. The attacker does not need to know that the target is a warrior. This is polymorphism.

**Mage**
* New attributes `mana` and `mana_max`, both with default value 30. Mana is stored in the protected attribute `_mana` and shown through the read-only property `mana`, in the same way as `hp`. A starting mana outside 0 to `mana_max` raises a `ValueError`.
* `attack()` is overridden:
    * With at least 10 mana, the mage casts a spell. It spends 10 mana and deals double the base `damage` (2 × 10 = 20 by default). The weapon is not used, even if the mage has one.
    * With less than 10 mana, the mage uses the normal `Character.attack()` through `super()`, so the weapon is used if one is equipped. Afterwards the mage restores 5 mana, but never more than `mana_max`.
* The spell cost (10) and the mana restored (5) are stored as the class constants `SPELL_COST` and `MANA_RESTORE`, so the numbers are not hidden inside the method.

Both classes also override `__repr__` and `__str__`, so the armor and mana are shown when the objects are printed.

In [8]:
random.seed(201)  # Fixed seed, so the output is the same every time the notebook is run

# --- Warrior: armor reduces incoming damage ---
warrior = Warrior("Conan")
print(warrior)
print(repr(warrior))
print("Is a Character:", isinstance(warrior, Character))
print("New attributes and methods compared to Character:",
      sorted(set(dir(warrior)) - set(dir(Character("Dummy")))))

print()
for incoming in [12, 5, 3]:
    hp_before = warrior.hp
    warrior.take_damage(incoming)
    print(f"{warrior.name} is hit for {incoming:2} damage: loses {hp_before - warrior.hp} HP ({warrior.hp} HP left)")

# A normal Character attacking the warrior: attack() calls the warrior's own take_damage()
goblin = Character("Goblin", damage=10)
hp_before = warrior.hp
goblin.attack(warrior)
print(f"{goblin.name} attacks with {goblin.damage} damage: {warrior.name} loses {hp_before - warrior.hp} HP")

# --- Mage: spells cost mana, base attacks restore mana ---
mage = Mage("Merlin", weapon=Weapon("Staff", Dice(sides=6)))
print()
print(mage)
print(repr(mage))
print("New attributes and methods compared to Character:",
      sorted(set(dir(mage)) - set(dir(Character("Dummy")))))

print()
dummy = Character("Dummy", hp_max=1000)
for attack_number in range(1, 10):
    mana_before = mage.mana
    hp_before = dummy.hp
    mage.attack(dummy)
    kind = "spell " if mage.mana < mana_before else "weapon"
    print(f"Attack {attack_number}: {kind} deals {hp_before - dummy.hp:2} damage, "
          f"mana {mana_before:2} -> {mage.mana:2}")

Warrior: Conan with 100 HP, 10 damage and 5 armor
<Warrior name=Conan hp=100 damage=10 weapon=None armor=5>
Is a Character: True
New attributes and methods compared to Character: ['armor']

Conan is hit for 12 damage: loses 7 HP (93 HP left)
Conan is hit for  5 damage: loses 0 HP (93 HP left)
Conan is hit for  3 damage: loses 0 HP (93 HP left)
Goblin attacks with 10 damage: Conan loses 5 HP

Mage: Merlin with 100 HP, 10 damage and 30/30 mana
<Mage name=Merlin hp=100 damage=10 weapon=Weapon(name=Staff, damage_dice=Dice(sides=6)) mana=30/30>
New attributes and methods compared to Character: ['MANA_RESTORE', 'SPELL_COST', '_mana', 'mana', 'mana_max']

Attack 1: spell  deals 20 damage, mana 30 -> 20
Attack 2: spell  deals 20 damage, mana 20 -> 10
Attack 3: spell  deals 20 damage, mana 10 ->  0
Attack 4: weapon deals  1 damage, mana  0 ->  5
Attack 5: weapon deals  6 damage, mana  5 -> 10
Attack 6: spell  deals 20 damage, mana 10 ->  0
Attack 7: weapon deals  3 damage, mana  0 ->  5
Attack 

**Observation.** The mage starts with 30 mana and casts three spells in a row (30 → 20 → 10 → 0), each dealing 20 damage. After that it falls into a fixed pattern: two weapon attacks restore 5 + 5 = 10 mana, which is exactly enough for one more spell. The staff only deals 1–6 damage, so the spells are where most of the mage's damage comes from. For the warrior, armor 5 cancels hits of 5 damage or less completely, and a base attack of 10 is halved.

### Task 4 – Tests
The tests check the default values, that both classes are subclasses of `Character`, that armor reduces damage without ever healing, that the mage casts spells with at least 10 mana and ignores the weapon, that the base attack uses the weapon and restores mana up to `mana_max`, that the mana pattern is correct over many attacks, that invalid values raise errors, and that `mana` is read-only.

In [9]:
# --- Warrior ---
warrior = Warrior("Warrior")
assert isinstance(warrior, Character) and warrior.armor == 5

# Armor is subtracted from incoming damage
warrior.take_damage(12)
assert warrior.hp == 93

# Damage smaller than or equal to the armor does nothing, and never heals
warrior.take_damage(5)
warrior.take_damage(3)
assert warrior.hp == 93

# HP still cannot go below 0
warrior.take_damage(1000)
assert warrior.hp == 0 and not warrior.is_alive

# Armor 0 gives the same result as a normal Character
w, c = Warrior("W", armor=0), Character("C")
w.take_damage(17)
c.take_damage(17)
assert w.hp == c.hp == 83

# Armor also works when another character attacks the warrior through attack()
warrior = Warrior("Warrior", armor=4)
Character("Attacker", damage=10).attack(warrior)
assert warrior.hp == 94

# Negative armor is not allowed
try:
    Warrior("W", armor=-1)
    raise AssertionError("negative armor should raise ValueError")
except ValueError:
    pass

# --- Mage ---
mage = Mage("Mage")
assert isinstance(mage, Character) and mage.mana == 30 and mage.mana_max == 30

# Spell with at least 10 mana: costs 10 mana and deals double base damage, even with a weapon equipped
mage = Mage("Mage", damage=10, weapon=Weapon("Stick", Dice(sides=2)))
target = Character("Target")
mage.attack(target)
assert target.hp == 80 and mage.mana == 20

# Exactly 10 mana is enough for a spell
mage = Mage("Mage", mana=10)
target = Character("Target")
mage.attack(target)
assert target.hp == 80 and mage.mana == 0

# Base attack with less than 10 mana: uses the weapon and restores 5 mana
mage = Mage("Mage", damage=10, weapon=Weapon("Stick", Dice(sides=2)), mana=9)
target = Character("Target")
mage.attack(target)
assert 1 <= 100 - target.hp <= 2 and mage.mana == 14

# Base attack without a weapon uses the base damage
mage = Mage("Mage", damage=7, mana=0)
target = Character("Target")
mage.attack(target)
assert target.hp == 93 and mage.mana == 5

# Restored mana never goes above mana_max
mage = Mage("Mage", mana=8, mana_max=10)
mage.attack(Character("Target"))
assert mage.mana == 10

# The mana pattern over many attacks: three spells, then two base attacks per spell
mage = Mage("Mage")
target = Character("Target", hp_max=10_000)
mana_after_each_attack = []
for _ in range(9):
    mage.attack(target)
    mana_after_each_attack.append(mage.mana)
assert mana_after_each_attack == [20, 10, 0, 5, 10, 0, 5, 10, 0]

# Starting mana must be between 0 and mana_max
for invalid_mana in [-1, 31]:
    try:
        Mage("Mage", mana=invalid_mana)
        raise AssertionError(f"mana={invalid_mana} should raise ValueError")
    except ValueError:
        pass

# mana is read-only
try:
    mage.mana = 100
    raise AssertionError("mana should be read-only")
except AttributeError:
    pass

print("All Task 4 tests passed.")

All Task 4 tests passed.


## Task 5: Running battles

Create a `Battle` class to manage turn-based combat between two `Character` objects.

* Initialize `Battle` with `hero: Character` and `enemy: Character`.
* Implement a `play_round()` method where `hero` attacks `enemy`, and if `enemy` is still alive, `enemy` attacks `hero`.
* Implement a `simulate_battle()` method with a `while` loop that runs rounds until one character is no longer alive.

### Task 5 – The Battle class
The `Battle` class is written in its own module, `battle.py`.

* `Battle(hero, enemy)` stores the two characters and a round counter, `rounds`.
* `play_round()` lets the hero attack the enemy. Only if the enemy is still alive does it attack back. This means the hero has an advantage: if both would die in the same round, the hero wins.
* `simulate_battle()` uses a `while` loop that plays rounds as long as both characters are alive, and returns the winner.
* `winner` is a read-only property that returns the character that is still alive, or `None` if both are alive.

`Battle` only uses `attack()` and `is_alive`, which all characters have. It therefore works with `Character`, `Warrior` and `Mage` without knowing which type each character is: the armor and mana rules are handled inside the characters' own methods.

**Our choice: a maximum number of rounds.** If neither character can hurt the other (for example two warriors whose armor is at least as high as the incoming damage), the `while` loop would never end. We therefore added `max_rounds` (default 1000). If this limit is reached, the battle stops and `simulate_battle()` returns `None`, meaning there was no winner.

In [10]:
random.seed(201)  # Fixed seed, so the output is the same every time the notebook is run

battle = Battle(Warrior("Conan"), Mage("Merlin"))
print(battle)
print("Attributes and methods:", [name for name in dir(battle) if not name.startswith("__")])

print()
winner = battle.simulate_battle(verbose=True)
print(f"\n{winner.name} wins after {battle.rounds} rounds!")
print(battle)

<Battle hero=Conan enemy=Merlin rounds=0>
Attributes and methods: ['enemy', 'hero', 'play_round', 'rounds', 'simulate_battle', 'winner']

Round 1: Conan 85 HP, Merlin 90 HP
Round 2: Conan 70 HP, Merlin 80 HP
Round 3: Conan 55 HP, Merlin 70 HP
Round 4: Conan 50 HP, Merlin 60 HP
Round 5: Conan 45 HP, Merlin 50 HP
Round 6: Conan 30 HP, Merlin 40 HP
Round 7: Conan 25 HP, Merlin 30 HP
Round 8: Conan 20 HP, Merlin 20 HP
Round 9: Conan 5 HP, Merlin 10 HP
Round 10: Conan 5 HP, Merlin 0 HP

Conan wins after 10 rounds!
<Battle hero=Conan enemy=Merlin rounds=10>


### Task 5 – Tests
The tests check that `play_round()` lets the hero attack first and only lets the enemy attack back if it is still alive, that `simulate_battle()` stops as soon as one character is dead and returns the right winner, that both the hero and the enemy can win, that `max_rounds` stops a battle where nobody takes damage, and that many random battles with all character types always end with exactly one survivor.

In [11]:
# play_round(): the hero attacks first, then the enemy attacks back
hero, enemy = Character("Hero", damage=10), Character("Enemy", damage=7)
battle = Battle(hero, enemy)
assert battle.winner is None and battle.rounds == 0
battle.play_round()
assert enemy.hp == 90 and hero.hp == 93 and battle.rounds == 1

# play_round(): a dead enemy does not attack back
hero, enemy = Character("Hero", damage=50), Character("Enemy", hp_max=50, damage=7)
Battle(hero, enemy).play_round()
assert not enemy.is_alive and hero.hp == 100

# simulate_battle(): fixed damage, so the result can be calculated by hand.
# The enemy has 35 HP and needs 4 hits. It attacks back in round 1-3 only: 3 x 5 = 15 damage.
hero, enemy = Character("Hero", damage=10), Character("Enemy", hp_max=35, damage=5)
battle = Battle(hero, enemy)
assert battle.simulate_battle() is hero
assert battle.rounds == 4 and hero.hp == 85 and enemy.hp == 0
assert battle.winner is hero

# The enemy can also win
hero, enemy = Character("Hero", hp_max=20, damage=1), Character("Enemy", damage=10)
assert Battle(hero, enemy).simulate_battle() is enemy
assert not hero.is_alive and enemy.is_alive

# max_rounds: two warriors whose armor blocks all damage would fight forever
hero = Warrior("Hero", damage=10, armor=10)
enemy = Warrior("Enemy", damage=10, armor=10)
battle = Battle(hero, enemy)
assert battle.simulate_battle(max_rounds=50) is None
assert battle.rounds == 50 and hero.hp == 100 and enemy.hp == 100

# Many random battles between all types, with and without weapons:
# every battle ends with exactly one survivor, and the winner is the one alive
def make_fighters() -> list[Character]:
    return [
        Character("Character"),
        Warrior("Warrior"),
        Mage("Mage"),
        Warrior("Warrior with axe", weapon=Weapon("Axe", Dice(sides=12))),
        Mage("Mage with staff", weapon=Weapon("Staff", Dice(sides=6))),
    ]

random.seed(1)
for _ in range(20):
    for hero in make_fighters():
        for enemy in make_fighters():
            battle = Battle(hero, enemy)
            winner = battle.simulate_battle()
            assert winner is not None and winner.is_alive
            assert hero.is_alive != enemy.is_alive
            hero.heal(hero.hp_max)  # Reset the hero's HP before the next battle

print("All Task 5 tests passed.")

All Task 5 tests passed.


## Simulated battles
Here the classes are put together in battles between different types of characters, with and without weapons. Each matchup is run 1000 times, and we count how often the hero (who attacks first) wins and how many rounds the battles last on average. A new pair of characters is created for every battle, so each battle starts with full HP and mana.

The weapons used are a sword (`Dice(sides=12)`, average 6.5 damage), a staff (`Dice(sides=6)`, average 3.5) and a greatsword (`Dice(sides=20)`, average 10.5).

In [12]:
random.seed(201)  # Fixed seed, so the output is the same every time the notebook is run

# One battle shown round by round: a warrior with a sword against a mage with a staff
print("Warrior with sword vs Mage with staff:")
battle = Battle(Warrior("Conan", weapon=Weapon("Sword", Dice(sides=12))),
                Mage("Merlin", weapon=Weapon("Staff", Dice(sides=6))))
winner = battle.simulate_battle(verbose=True)
print(f"{winner.name} wins after {battle.rounds} rounds!\n")


def run_battles(make_hero, make_enemy, n_battles: int = 1000) -> tuple[float, float]:
    """Run many battles and return (share of battles won by the hero, average number of rounds).

    make_hero and make_enemy are functions that create a new character, so every
    battle starts with fresh characters.
    """
    hero_wins = 0
    total_rounds = 0
    for _ in range(n_battles):
        battle = Battle(make_hero(), make_enemy())
        if battle.simulate_battle() is battle.hero:
            hero_wins += 1
        total_rounds += battle.rounds
    return hero_wins / n_battles, total_rounds / n_battles


# Each matchup: (hero description, function creating the hero, enemy description, function creating the enemy)
matchups = [
    # Without weapons
    ("Character", lambda: Character("Hero"), "Character", lambda: Character("Enemy")),
    ("Warrior", lambda: Warrior("Hero"), "Character", lambda: Character("Enemy")),
    ("Warrior", lambda: Warrior("Hero"), "Mage", lambda: Mage("Enemy")),
    ("Mage", lambda: Mage("Hero"), "Warrior", lambda: Warrior("Enemy")),
    # With weapons
    ("Warrior with sword", lambda: Warrior("Hero", weapon=Weapon("Sword", Dice(sides=12))),
     "Mage", lambda: Mage("Enemy")),
    ("Warrior", lambda: Warrior("Hero"),
     "Mage with staff", lambda: Mage("Enemy", weapon=Weapon("Staff", Dice(sides=6)))),
    ("Warrior with sword", lambda: Warrior("Hero", weapon=Weapon("Sword", Dice(sides=12))),
     "Mage with staff", lambda: Mage("Enemy", weapon=Weapon("Staff", Dice(sides=6)))),
    ("Warrior with greatsword", lambda: Warrior("Hero", weapon=Weapon("Greatsword", Dice(sides=20))),
     "Mage with staff", lambda: Mage("Enemy", weapon=Weapon("Staff", Dice(sides=6)))),
]

print(f"{'Hero':<25} {'Enemy':<18} {'Hero wins':>10} {'Avg rounds':>11}")
for hero_name, make_hero, enemy_name, make_enemy in matchups:
    win_share, avg_rounds = run_battles(make_hero, make_enemy)
    print(f"{hero_name:<25} {enemy_name:<18} {win_share:>10.1%} {avg_rounds:>11.1f}")

Warrior with sword vs Mage with staff:
Round 1: Conan 85 HP, Merlin 98 HP
Round 2: Conan 70 HP, Merlin 86 HP
Round 3: Conan 55 HP, Merlin 80 HP
Round 4: Conan 55 HP, Merlin 73 HP
Round 5: Conan 55 HP, Merlin 68 HP
Round 6: Conan 40 HP, Merlin 56 HP
Round 7: Conan 39 HP, Merlin 44 HP
Round 8: Conan 39 HP, Merlin 36 HP
Round 9: Conan 24 HP, Merlin 31 HP
Round 10: Conan 24 HP, Merlin 23 HP
Round 11: Conan 24 HP, Merlin 14 HP
Round 12: Conan 9 HP, Merlin 6 HP
Round 13: Conan 9 HP, Merlin 3 HP
Round 14: Conan 9 HP, Merlin 0 HP
Conan wins after 14 rounds!

Hero                      Enemy               Hero wins  Avg rounds
Character                 Character              100.0%        10.0
Warrior                   Character              100.0%        10.0
Warrior                   Mage                   100.0%        10.0
Mage                      Warrior                100.0%        10.0
Warrior with sword        Mage                     0.1%        10.0
Warrior                   Mage with

**Discussion of the simulated battles.**
* *Without weapons the battles are fully predictable.* No dice are rolled, so the same matchup gives the same result every time (0 % or 100 %). When the two characters are evenly matched, the one who attacks first wins: the warrior beats the mage as the hero, but the mage beats the warrior when the mage is the hero. In both cases the loser would have needed only one more round. This shows the advantage the hero gets from `play_round()`, where a dead enemy cannot attack back.
* *A weapon is only an improvement if its dice beats the base damage.* The sword (12-sided dice, average 6.5) is weaker than the base damage of 10, so equipping it makes the warrior go from winning 100 % to 0.1 % against the mage. The greatsword (20-sided dice, average 10.5) is slightly better than the base damage, and the warrior wins 99.6 %.
* *Armor works best against many small hits.* Armor removes a fixed amount (5) from every hit. The staff (6-sided dice) therefore does almost nothing against the warrior: only a roll of 6 gets through, and then only 1 damage. The mage's spells are not affected by the staff, because spells never use the weapon, so most of the mage's damage comes from the spells.
* *Randomness makes battles closer and longer.* The only evenly matched battle is the warrior with a sword against the mage with a staff (40.2 %), and it also lasts the longest (14.6 rounds on average), since both characters deal less damage per hit than without weapons.